# Data Visualization with Matplotlib

Last week, we collected book titles and prices from a website and saved the results in `books.csv`.

In this lesson, we will use that dataset to create charts with Matplotlib. We will start with a simple plot, explore common chart types, customize their appearance, and save the figures as PNG images.

Run the cells from top to bottom. The notebook saves chart images in the `images/` folder.

## 1. Import the Libraries

We will use `pandas` to load and organize tabular data and `matplotlib.pyplot` to create charts. The aliases `pd` and `plt` are conventional short names used throughout the notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Store all chart images in one folder.
IMAGE_DIR = Path("images")
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the Dataset

The `books.csv` file was created in the previous lesson. This cell checks that the file exists before loading it into a pandas DataFrame named `df`.

Keep `books.csv` in the same working directory as this notebook.

In [ ]:
file_path = Path("books.csv")

if not file_path.exists():
    raise FileNotFoundError(
        "books.csv was not found. Place it in the notebook's working directory."
    )

df = pd.read_csv(file_path)
df.head()

### Inspect the Dataset

Before plotting, check the number of rows and columns, the column names, and the data types. These checks help us understand what kind of values are available.

In [ ]:
print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())
df.info()

### Check the Required Columns

The examples in this notebook use the `title` and `price` columns. This check provides a clear error message if the CSV has a different structure.

In [ ]:
required_columns = {"title", "price"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print("Required columns are available.")

## 3. Prepare the Price Data

Prices collected from a website may include currency symbols, such as `£51.77`. To plot them numerically, we will remove characters other than digits and decimal points, then convert the values to numbers.

Invalid values become missing values (`NaN`), which we will remove before plotting.

In [ ]:
df["price"] = pd.to_numeric(
    df["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True),
    errors="coerce"
)

df = df.dropna(subset=["title", "price"]).copy()
df["title"] = df["title"].astype(str).str.strip()
df = df[df["title"] != ""].reset_index(drop=True)

if df.empty:
    raise ValueError("No valid book titles and prices remain after cleaning.")

print("Rows available for plotting:", len(df))
df[["title", "price"]].head()

### Verify the Prepared Data

The `price` column should now contain numeric values. We will check its data type and confirm that there are no missing prices left in the cleaned dataset.

In [ ]:
print(df[["title", "price"]].dtypes)
print("Missing prices:", df["price"].isna().sum())
print("Number of books:", len(df))

## 4. Basic Chart Terminology

- **Figure:** The overall drawing area.
- **Axes:** The area where data is plotted.
- **X-axis:** The horizontal axis.
- **Y-axis:** The vertical axis.
- **Data point:** A value represented on a chart.
- **Title:** A short description of the chart.
- **Axis label:** Text explaining what an axis represents.

Titles and labels help readers understand a chart without needing additional explanation.

### Create a Simple Plot

`x` contains horizontal coordinates and `y` contains the corresponding vertical values. `plt.plot()` creates the plot, and `plt.show()` displays it.

First, we will create a basic plot without additional formatting.

In [ ]:
x = [1, 2, 3, 4, 5]
y = [2, 4, 3, 7, 6]

plt.plot(x, y)
plt.savefig(IMAGE_DIR / "first_line_plot.png", dpi=300, bbox_inches="tight")
plt.show()

### Add a Title and Axis Labels

`plt.title()` adds a title. `plt.xlabel()` and `plt.ylabel()` describe the horizontal and vertical axes. Clear labels make the meaning of the plotted values easier to understand.

In [ ]:
plt.plot(x, y, marker="o")

plt.title("A Simple Line Plot")
plt.xlabel("X Values")
plt.ylabel("Y Values")

plt.tight_layout()
plt.savefig(IMAGE_DIR / "labeled_line_plot.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Line Plot with Book Prices

A line plot connects data points in sequence. It is useful when the order of observations matters, such as measurements taken over time.

Here, we will plot the prices of the first 20 books. The horizontal axis represents each book's position in the dataset, not time, so the chart should not be interpreted as a time trend.

In [ ]:
sample = df.head(20)

plt.figure(figsize=(9, 5))
plt.plot(sample["price"], marker="o")

plt.title("Prices of the First 20 Books")
plt.xlabel("Book Position")
plt.ylabel("Price")

plt.tight_layout()
plt.savefig(IMAGE_DIR / "book_prices_line.png", dpi=300, bbox_inches="tight")
plt.show()

### Markers and Line Styles

The `marker` parameter adds a symbol at each data point. The `linestyle` parameter controls how the points are connected. These options can make individual observations easier to see.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(sample["price"], marker="o", linestyle="--", label="Book price")

plt.title("Book Prices with Markers")
plt.xlabel("Book Position")
plt.ylabel("Price")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGE_DIR / "book_prices_line_styled.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Bar Chart

A bar chart compares numerical values across categories. Each bar represents one category, and its height represents the corresponding value.

We will compare the prices of the first 10 books. Since book titles can be long, the category labels are rotated to make them easier to read.

In [ ]:
sample10 = df.head(10)

plt.figure(figsize=(11, 6))
plt.bar(sample10["title"], sample10["price"])

plt.title("Prices of the First 10 Books")
plt.xlabel("Book")
plt.ylabel("Price")
plt.xticks(rotation=90)

plt.tight_layout()
plt.savefig(IMAGE_DIR / "book_prices_bar.png", dpi=300, bbox_inches="tight")
plt.show()

### Horizontal Bar Chart

A horizontal bar chart is often easier to read when categories have long names. `plt.barh()` creates horizontal bars. Sorting the sample by price makes the displayed values easier to compare.

In [ ]:
sample_sorted = df.head(10).sort_values("price")

plt.figure(figsize=(11, 6))
plt.barh(sample_sorted["title"], sample_sorted["price"])

plt.title("Prices of the First 10 Books")
plt.xlabel("Price")
plt.ylabel("Book")

plt.tight_layout()
plt.savefig(IMAGE_DIR / "book_prices_horizontal_bar.png", dpi=300, bbox_inches="tight")
plt.show()

### When Should We Use a Bar Chart?

Use a bar chart when comparing separate categories, such as prices of individual books, sales by product, or the number of students in different groups.

A line plot emphasizes a sequence, while a bar chart emphasizes comparisons between categories.

## 7. Scatter Plot

A scatter plot represents the relationship between two numerical variables. Each point corresponds to one observation.

The book dataset contains titles and prices, so it does not provide two numerical variables for this example. We will use illustrative data about study hours and exam scores instead. These values are examples, not real student records.

### Prepare the Example Data

Each value in `study_hours` is paired with the value at the same position in `exam_scores`. For example, the first pair represents 1 study hour and an exam score of 45.

In [ ]:
study_hours = [1, 2, 2.5, 3, 4, 5, 6, 7]
exam_scores = [45, 50, 55, 60, 65, 75, 82, 90]

### Create the Scatter Plot

`plt.scatter()` places one point for each pair of values. The axes show the two numerical variables. A scatter plot can help us visually inspect how variables vary together, but it does not by itself establish cause and effect.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(study_hours, exam_scores)

plt.title("Study Hours and Exam Scores")
plt.xlabel("Study Hours")
plt.ylabel("Exam Score")

plt.tight_layout()
plt.savefig(IMAGE_DIR / "study_hours_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

### When Should We Use a Scatter Plot?

Use a scatter plot when each observation has two numerical measurements, such as height and weight or advertising spend and sales. Each point shows both values for one observation.

## 8. Histogram

A histogram shows the distribution of numerical values by grouping them into intervals called **bins**. The height of each bar represents how many observations fall within that interval.

We will use book prices to see how frequently different price ranges occur.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["price"], bins=10, edgecolor="black")

plt.title("Distribution of Book Prices")
plt.xlabel("Price")
plt.ylabel("Frequency")

plt.tight_layout()
plt.savefig(IMAGE_DIR / "book_price_histogram.png", dpi=300, bbox_inches="tight")
plt.show()

### Change the Number of Bins

The `bins` parameter controls how many intervals are used. Fewer bins create broader intervals; more bins create narrower intervals and may show more detail. The best choice depends on the data and the purpose of the chart.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["price"], bins=5, edgecolor="black")

plt.title("Book Price Distribution: 5 Bins")
plt.xlabel("Price")
plt.ylabel("Frequency")

plt.tight_layout()
plt.savefig(IMAGE_DIR / "book_price_histogram_5_bins.png", dpi=300, bbox_inches="tight")
plt.show()

### Bar Chart vs. Histogram

A bar chart compares values across separate categories, such as individual book titles. A histogram groups numerical values into intervals, such as price ranges.

Histogram bars usually touch because the intervals represent adjacent ranges on a numerical scale.

## 9. Customize Charts

Matplotlib offers options to improve readability:

- `figsize` controls the figure size.
- `marker` changes how data points appear.
- `grid()` adds reference lines.
- `legend()` displays labels for data series.
- `tight_layout()` adjusts spacing to reduce overlap.

We will combine several of these options in one chart.

In [ ]:
sample = df.head(10)

plt.figure(figsize=(10, 5))
plt.plot(sample["price"], marker="o", linestyle="-", label="Book price")

plt.title("Book Prices")
plt.xlabel("Book Position")
plt.ylabel("Price")
plt.xticks(range(len(sample)), range(1, len(sample) + 1))
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.savefig(IMAGE_DIR / "customized_chart.png", dpi=300, bbox_inches="tight")
plt.show()

### Plot Multiple Data Series

A chart can contain multiple data series. Each series can have its own label, and the legend helps readers distinguish them.

This example uses illustrative values to demonstrate the technique.

In [ ]:
days = [1, 2, 3, 4, 5]
series_a = [3, 5, 4, 7, 8]
series_b = [2, 3, 5, 5, 7]

plt.figure(figsize=(8, 5))
plt.plot(days, series_a, marker="o", label="Series A")
plt.plot(days, series_b, marker="s", label="Series B")

plt.title("Comparing Two Data Series")
plt.xlabel("Day")
plt.ylabel("Value")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGE_DIR / "multiple_series.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Save a Chart as a PNG Image

`plt.savefig()` saves the current figure to a file. In this notebook, all figures are stored in the `images/` folder.

The `dpi` parameter controls image resolution. `bbox_inches="tight"` helps include chart elements without excessive empty space. Save the figure before calling `plt.show()`.

In [ ]:
sample = df.head(10).sort_values("price")

plt.figure(figsize=(10, 6))
plt.barh(sample["title"], sample["price"])

plt.title("Prices of the First 10 Books")
plt.xlabel("Price")
plt.ylabel("Book")

plt.tight_layout()
plt.savefig(IMAGE_DIR / "saved_book_prices.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved image:", IMAGE_DIR / "saved_book_prices.png")